In [ ]:
import numpy as np
import qutip as qt

In [ ]:
def get_qubit_ham(N, wm=1.0, fixed_seed=False, indv_qubit=False, ds_dis=0.0, sigz_ham=False):
    if fixed_seed:
        np.random.seed(0)

    ds = np.random.uniform(-ds_dis, ds_dis, N)
    ds += 1.0

    sigz = qt.sigmaz()
    sigy = qt.sigmay()
    sigx = qt.sigmax()
    eye = qt.qeye(2)

    eyeList = [eye] * N

    qH0 = 0
    qH1 = 0
    qH1_list = []

    for i in range(N):
        ops0 = eyeList.copy()
        ops1 = eyeList.copy()

        if sigz_ham:
            ops0[i] = -0.5 * wm * sigz
            ops1[i] = ds[i] * sigx
        else:
            ops0[i] = -0.5 * wm * sigx
            ops1[i] = ds[i] * sigz

        qH0 += qt.tensor(ops0)

        if not indv_qubit:
            qH1 += qt.tensor(ops1)
        else:
            qH1_list.append(qt.tensor(ops1))

    eigenvalues, eigenstates = qH0.eigenstates()

    if not indv_qubit:
        return qH0, qH1, eigenvalues, eigenstates
    else:
        return qH0, qH1_list, eigenvalues, eigenstates
    
def get_dis_qubit_ham(qH0_dis, N, N_dis=None, ham_disorder=[0, 0, 0], fixed_seed=False):
    if N_dis == None:
        N_dis = N

    if fixed_seed:
        np.random.seed(0)

    if ham_disorder[0] != 0.0:
        zd = ham_disorder[0]
        hz = np.zeros(N)
        dis_sites = np.random.choice(N, size=N_dis, replace=False)
        hz[dis_sites] = np.random.uniform(-zd, zd, N_dis)

    if ham_disorder[1] != 0.0:
        yd = ham_disorder[1]
        hy = np.zeros(N)
        dis_sites = np.random.choice(N, size=N_dis, replace=False)
        hy[dis_sites] = np.random.uniform(-yd, yd, N_dis)

    if ham_disorder[2] != 0.0:
        xd = ham_disorder[2]
        hx = np.zeros(N)
        dis_sites = np.random.choice(N, size=N_dis, replace=False)
        hx[dis_sites] = np.random.uniform(-xd, xd, N_dis)

    sigz = qt.sigmaz()
    sigy = qt.sigmay()
    sigx = qt.sigmax()
    eye = qt.qeye(2)

    eyeList = [eye] * N

    ham_dis = qt.Qobj(np.zeros((2**N, 2**N)), dims=[[2]*N, [2]*N])

    for i in range(N):
        zops0 = eyeList.copy()
        yops0 = eyeList.copy()
        xops0 = eyeList.copy()

        if ham_disorder[0] != 0.0:
            zops0[i] = hz[i] * sigz

        if ham_disorder[1] != 0.0:
            yops0[i] = hy[i] * sigy

        if ham_disorder[2] != 0.0:
            xops0[i] = hx[i] * sigx
        
        ham_dis += qt.tensor(zops0)
        ham_dis += qt.tensor(yops0)
        ham_dis += qt.tensor(xops0)

    qH0_dis += ham_dis

    qeigenvalues, qeigenstates = qH0_dis.eigenstates()

    return qH0_dis, qeigenvalues, qeigenstates